### Recursive Character Splitter

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from dotenv import load_dotenv
import os
load_dotenv()

text = """
LangChain is a framework for developing applications powered by language models.
It enables context-aware and reasoning-based applications.
"""
# trying a different text to see how the overlap works when the text is split at a clean boundary (newline).
text1 = "It looks like the overlap isn't showing up because your input text is too short and naturally splits at a clean boundary (the newline).The RecursiveCharacterTextSplitter tries to split on double newlines (\n\n), then single newlines (\n), then spaces. In your code:Chunk 1 is LangChain is a framework... models.(approx. 78 characters). This fits well under your chunk_size=100.The splitter then looks at the remaining text: It enables context-aware... applications. (approx. 57 characters).Because the first sentence plus the second sentence combined would exceed 100 characters, it keeps them separate.Why there is no visible overlap:The splitter only applies the chunk_overlap when it is forced to break a large block of text into pieces. Since your sentences are already separated by a newline (\n) and each is smaller than 100 characters, the splitter treats them as two distinct logical units.If it were to overlap them, it would look like this:Chunk 1: [Sentence A]Chunk 2: [Last 50 chars of Sentence A] + [Sentence B]However, the recursive splitter prioritizes keeping sentences whole. It won't grab 50 characters from the middle of Sentence A just to satisfy the overlap rule if the sentences already fit within the size limit."

text_splitter = RecursiveCharacterTextSplitter(
    # we can put cutom seperators here to as we want the text to be split at those points. By default it looks for \n\n, then \n, then space.
    #example custom seperators: separators=["\n\n", "\n", ".", "!", "?", " ", ""]
    chunk_size=100,
    chunk_overlap=10
)

chunks = text_splitter.split_text(text)

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}:\n{chunk}\n")

vectordb = Chroma.from_texts(
    chunks,
    OpenAIEmbeddings(
        model="text-embedding-3-small",
        api_key=os.getenv("API_KEY"),
        base_url=os.getenv("BASE_URL"),
    ),
)

print(vectordb._collection.get(include=['embeddings']))

Chunk 1:
LangChain is a framework for developing applications powered by language models.

Chunk 2:
It enables context-aware and reasoning-based applications.

{'ids': ['b3966851-5924-401b-ae49-31f4347f3f35', '5d5843ee-d7ba-40de-bb2d-05429a4fb283'], 'embeddings': array([[-0.03170776,  0.0093689 ,  0.04568481, ..., -0.04559326,
        -0.01431274, -0.00512695],
       [ 0.00347328,  0.01248169,  0.04141235, ..., -0.01870728,
        -0.01600647,  0.02168274]], shape=(2, 1536)), 'documents': None, 'uris': None, 'included': ['embeddings'], 'data': None, 'metadatas': None}


In [15]:
# returns ids plus default docs/metadatas
result = vectordb._collection.get()
ids = result["ids"]
vectordb._collection.delete(ids=ids)

{'deleted': 2}

In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = """LangChain is a powerful framework for developing applications powered by language models.
It enables developers to chain together components like LLMs, prompts, and memory to create advanced conversational AI systems.
Text splitters in LangChain help break large documents into smaller pieces for processing."""

splitter = RecursiveCharacterTextSplitter(
    chunk_size=80,
    chunk_overlap=20,
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = splitter.split_text(text)

print("📄 Number of Chunks:", len(chunks))
for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1}:\n{chunk}")

📄 Number of Chunks: 9

Chunk 1:
LangChain is a powerful framework for developing applications powered by

Chunk 2:
powered by language models

Chunk 3:
.

Chunk 4:
It enables developers to chain together components like LLMs, prompts, and

Chunk 5:
LLMs, prompts, and memory to create advanced conversational AI systems

Chunk 6:
.

Chunk 7:
Text splitters in LangChain help break large documents into smaller pieces for

Chunk 8:
smaller pieces for processing

Chunk 9:
.


### Character Text Splitter

In [16]:
from langchain_text_splitters import CharacterTextSplitter

text = """LangChain is a powerful framework for developing applications powered by language models.
It enables developers to chain together components like LLMs, prompts, and memory to create advanced conversational AI systems.
Text splitters in LangChain help break large documents into smaller pieces for processing."""

splitter = CharacterTextSplitter(
    chunk_size=40,
    chunk_overlap=10,
    separator=" "
)

chunks = splitter.split_text(text.replace("\n", " "))

print("📄 Number of Chunks:", len(chunks))
for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1}:\n{chunk}")

📄 Number of Chunks: 10

Chunk 1:
LangChain is a powerful framework for

Chunk 2:
for developing applications powered by

Chunk 3:
powered by language models. It enables

Chunk 4:
It enables developers to chain together

Chunk 5:
together components like LLMs, prompts,

Chunk 6:
prompts, and memory to create advanced

Chunk 7:
advanced conversational AI systems. Text

Chunk 8:
Text splitters in LangChain help break

Chunk 9:
help break large documents into smaller

Chunk 10:
smaller pieces for processing.


### TokenTextSplitter

In [18]:
from langchain_text_splitters import TokenTextSplitter

text = """LangChain simplifies working with LLMs by providing modular components like prompts, chains, memory, and tools."""

splitter = TokenTextSplitter(
    chunk_size=10,
    chunk_overlap=2
)

chunks = splitter.split_text(text)

print("📄 Number of Chunks:", len(chunks))
for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1}:\n{chunk}")

📄 Number of Chunks: 3

Chunk 1:
LangChain simplifies working with LLMs by

Chunk 2:
Ms by providing modular components like prompts, chains,

Chunk 3:
 chains, memory, and tools.


### MarkdownTextSplitter

In [19]:
from langchain_text_splitters import MarkdownTextSplitter

md_text = """
# LangChain Overview
LangChain helps developers build applications using LLMs.

## Components
- LLMs
- Chains
- Memory
- Agents
"""

splitter = MarkdownTextSplitter(chunk_size=100, chunk_overlap=10)
chunks = splitter.split_text(md_text)

for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1}:\n{chunk}")


Chunk 1:
# LangChain Overview
LangChain helps developers build applications using LLMs.

Chunk 2:
## Components
- LLMs
- Chains
- Memory
- Agents
